# Q-learning on Taxi

Same algorithm as in Example 02 (**Q-learning**), now on a richer problem: the
[Taxi environment](https://gymnasium.farama.org/environments/toy_text/taxi/) from Gymnasium.

$$Q(s,a) \leftarrow Q(s,a) - \alpha \Big[\, Q(s,a) - \big( r + \gamma\, \max_{a'} Q(s',a') \big) \Big]$$

**The task.** A taxi drives on a 5x5 map. It must go to the passenger, pick them up, drive to the destination and drop them off.

**What is new compared with FrozenLake?**
- The state is **not only the position** of the taxi. It also includes where the passenger is and where they want to go:
  25 taxi positions × 5 passenger locations (R, G, Y, B or *in the taxi*) × 4 destinations = **500 states**.
  The same square needs a different action depending on the passenger!
- There are 6 actions: 4 moves + **pickup** + **dropoff**.
- Rewards: −1 per step, −10 for an illegal pickup/dropoff, +20 for a successful delivery (the episode ends). Episodes are cut at 200 steps.

In [ ]:
# Uncomment if needed (tested with Gymnasium 1.3.0).
# %pip install "gymnasium>=1.3" matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

print("Gymnasium version:", gym.__version__)

## 1. The environment

In the map, `|` are walls, `R G Y B` are the four stations, the yellow cursor is the taxi,
the **blue** letter is the passenger's position and the **magenta** letter is the destination.

In [ ]:
# ------------------------------------------------------------------ #
# Environment                                                        #
# ------------------------------------------------------------------ #
env = gym.make("Taxi-v4", render_mode="ansi")

N         = env.observation_space.n       # 500 states
N_ACTIONS = env.action_space.n            # 6 actions
GAMMA     = 0.9

SOUTH, NORTH, EAST, WEST, PICKUP, DROPOFF = 0, 1, 2, 3, 4, 5
A_NAMES = ["SOUTH", "NORTH", "EAST", "WEST", "PICKUP", "DROPOFF"]
LOC_NAMES = ["R", "G", "Y", "B", "in taxi"]     # passenger locations (index 0..4)

print("Number of states :", N)
print("Number of actions:", N_ACTIONS)
s, _ = env.reset(seed=0)
print(env.render())

## 2. What is inside a state?

A state is a single integer (0..499), so the Q-table is still `Q[s, a]`.
`env.unwrapped.decode(s)` tells us what the number means.

In [ ]:
taxi_row, taxi_col, passenger, destination = env.unwrapped.decode(s)
print(f"state {s}:  taxi at (row={taxi_row}, col={taxi_col}),  "
      f"passenger at {LOC_NAMES[passenger]},  destination {LOC_NAMES[destination]}")

# Interacting is the same as in FrozenLake
sp, r, terminated, truncated, info = env.step(NORTH)
print(f"a=NORTH -> s'={sp}  r={r}  terminated={terminated}  truncated={truncated}")
sp, r, terminated, truncated, info = env.step(PICKUP)      # no passenger here → illegal
print(f"a=PICKUP -> s'={sp}  r={r}  (illegal pickup!)")

## 3. ε-greedy action selection

Same as before (ties are broken randomly).

In [ ]:
# ------------------------------------------------------------------ #
# Epsilon-greedy action selection                                    #
# ------------------------------------------------------------------ #
def epsilon_greedy(Q, s, eps, rng):
    """
    With probability eps    → pick a random action (explore)
    With probability 1-eps  → pick argmax_a Q(s,a)  (exploit)
    """
    if rng.random() < eps:
        return int(rng.integers(0, N_ACTIONS))
    best_actions = np.flatnonzero(Q[s] == Q[s].max())   # all maximal actions
    return int(rng.choice(best_actions))                # break ties randomly

## 4. Q-learning

The code is the same as in Example 02. The only change: here **ε also decays** (from 1.0 to 0.05).
With 500 states a fully random behaviour (ε = 1) still works, because Q-learning is off-policy,
but it needs much more time (see *Try this*).

✏️ **Exercise.** Complete the two lines marked `# TODO` with the Q-learning update (the same as in Example 02):

$$\delta = Q(s,a) - \big(r + \gamma\, \max_{a'} Q(s',a')\big), \qquad Q(s,a) \leftarrow Q(s,a) - \alpha\,\delta$$

*Hint:* `np.max(Q[sp])` gives $\max_{a'} Q(s',a')$.

In [ ]:
# ------------------------------------------------------------------ #
# Q-learning                                                         #
# ------------------------------------------------------------------ #
def q_learning(
    env,
    n_episodes  = 5000,
    alpha_0     = 0.5,      # initial learning rate
    eps_0       = 1.0,      # initial epsilon
    eps_min     = 0.05,     # floor for epsilon
    alpha_min   = 0.01,     # floor for alpha
    decay_every = 200,      # decay period (episodes)
    decay_rate  = 0.90,     # multiplicative decay factor
    seed        = 42,
):
    rng = np.random.default_rng(seed)
    Q   = np.zeros((N, N_ACTIONS))      # action-value table: 500 x 6
    env.reset(seed=seed)

    eps   = eps_0
    alpha = alpha_0

    ep_returns  = []   # sum of rewards per episode (for plotting)
    ep_lengths  = []   # steps per episode

    for ep in range(n_episodes):
        if (ep%1000 == 0):
            print(f"episode: {ep} done")

        # ---- decay schedule ----------------------------------------
        if ep > 0 and ep % decay_every == 0:
            eps   = max(eps   * decay_rate, eps_min)
            alpha = max(alpha * decay_rate, alpha_min)

        # ---- start of episode: random taxi, passenger, destination --
        s, _ = env.reset()
        a    = epsilon_greedy(Q, s, eps, rng)

        total_r = 0
        steps   = 0

        # ---- Q-learning update loop --------------------------------
        while True:
            sp, r, terminated, truncated, _ = env.step(a)

            # Note: a delivery ends the episode (terminal). Q is never updated
            # there, so np.max(Q[sp]) = 0 when sp is terminal.
            # Core Q-learning update: target uses max over Q(s',·)
            td_error  = None                     # TODO: complete (off-policy target)
            Q[s, a]   = Q[s, a] - None           # TODO: complete

            total_r += r
            steps   += 1
            s = sp
            a = epsilon_greedy(Q, s, eps, rng)        # behaviour policy

            if terminated or truncated:               # end of episode
                break

        ep_returns.append(total_r)
        ep_lengths.append(steps)

    return Q, ep_returns, ep_lengths

## 5. Run Q-learning

In [ ]:
# ------------------------------------------------------------------ #
# Run Q-learning                                                     #
# ------------------------------------------------------------------ #
print("Running Q-learning …")
Q, ep_returns, ep_lengths = q_learning(env)
print("Done!")

## 6. Results

At the start the agent behaves randomly: about 200 steps and many illegal pickups/dropoffs (return around −700).
As ε decreases, it learns to deliver the passenger in a few steps (return close to +8).

In [ ]:
# ------------------------------------------------------------------ #
# Visualisation returns                                              #
# ------------------------------------------------------------------ #
fig, axes = plt.subplots(1, 1, figsize=(7, 4))

ax = axes
window = 100
moving_avg = np.convolve(ep_returns, np.ones(window) / window, mode="valid")
ax.plot(ep_returns, color="tab:blue", linewidth=0.5, alpha=0.3, label="return")
ax.plot(range(window - 1, len(ep_returns)), moving_avg, color="tab:blue",
        linewidth=2.0, label=f"moving average ({window} episodes)")
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("Return")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Test the greedy policy

We play 100 new episodes (random starts) with the greedy policy $\pi^*(s) = \arg\max_a Q(s,a)$, without exploration.

In [ ]:
# ------------------------------------------------------------------ #
# Evaluate the greedy policy                                         #
# ------------------------------------------------------------------ #
n_test    = 100
successes = 0
steps     = []
for i in range(n_test):
    s, _ = env.reset(seed=1000 + i)
    for t in range(200):
        s, r, terminated, truncated, _ = env.step(int(np.argmax(Q[s])))
        if terminated or truncated:
            break
    successes += terminated
    steps.append(t + 1)

print(f"Successful deliveries: {successes}/{n_test}")
print(f"Average number of steps: {np.mean(steps):.1f}")

### Looking at the policy

The policy depends on the passenger, so we cannot draw a single map of arrows.
Below we draw **two slices** of the same Q-table, both with destination **B**:

- **left:** the passenger is waiting at **R** → the taxi should drive to R and pick up (`P`);
- **right:** the passenger is already in the taxi → the taxi should drive to B and drop off (`D`).

Thick black lines are walls.

In [ ]:
# ------------------------------------------------------------------ #
# Visualisation policy (two slices of the Q-table)                   #
# ------------------------------------------------------------------ #
greedy_policy = np.argmax(Q, axis=1)

LOCS = env.unwrapped.locs                   # [(0,0), (0,4), (4,0), (4,3)] = R, G, Y, B
DESC = env.unwrapped.desc.astype(str)       # the text map (used to find the walls)
dx   = { SOUTH:(0,.35), NORTH:(0,-.35), EAST:(.35,0), WEST:(-.35,0) }

def plot_policy_slice(ax, passenger, destination, title):
    # stations: passenger in green, destination in blue
    for i, (r, c) in enumerate(LOCS):
        color = "#00A2E8" if i == destination else ("#7FD17F" if i == passenger else "none")
        ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, color=color, alpha=0.6))
        ax.text(c - .45, r - .25, LOC_NAMES[i], fontsize=10, weight="bold")
    # walls: a '|' in the text map between two columns
    for r in range(5):
        for c in range(4):
            if DESC[r + 1, 2 * c + 2] == "|":
                ax.plot([c + .5, c + .5], [r - .5, r + .5], color="black", lw=4)
    # greedy action in every taxi position
    for r in range(5):
        for c in range(5):
            s = env.unwrapped.encode(r, c, passenger, destination)
            a = greedy_policy[s]
            if a == PICKUP or a == DROPOFF:
                ax.text(c, r, "P" if a == PICKUP else "D", ha="center", va="center",
                        fontsize=14, weight="bold", color="red")
            else:
                ddx, ddy = dx[a]
                ax.annotate("", xy=(c+ddx, r+ddy), xytext=(c-ddx, r-ddy),
                            arrowprops=dict(arrowstyle="-|>", color="green", lw=1.8))
    ax.set_xlim(-.5, 4.5); ax.set_ylim(4.5, -.5); ax.set_aspect("equal")
    ax.set_xticks(np.arange(-.5, 5), minor=True); ax.set_yticks(np.arange(-.5, 5), minor=True)
    ax.grid(which="minor", color="lightgray"); ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
plot_policy_slice(axes[0], passenger=0, destination=3, title="Passenger waiting at R, destination B")
plot_policy_slice(axes[1], passenger=4, destination=3, title="Passenger in the taxi, destination B")
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------------ #
# Q-values of one state                                              #
# ------------------------------------------------------------------ #
s = env.unwrapped.encode(0, 0, 0, 3)        # taxi at R, passenger at R, destination B
print("Taxi at R, passenger at R, destination B:\n")
for a in range(N_ACTIONS):
    print(f"  Q(s, {A_NAMES[a]:7s}) = {Q[s, a]:6.2f}")
print("\nGreedy action:", A_NAMES[np.argmax(Q[s])])

## 7. Save the policy

We save the whole Q-table (500 x 6). Then run the policy from a terminal, inside the `Lesson08` folder:

```
python 03_run_policy_taxi.py                 # 1 episode
python 03_run_policy_taxi.py --episodes 3    # 3 episodes
```

The script opens a window with the taxi (it needs `pip install "gymnasium[toy-text]"`).

In [ ]:
# ------------------------------------------------------------------ #
# Save the learned Q-table (used by 03_run_policy_taxi.py)           #
# ------------------------------------------------------------------ #
np.save("policy_taxi.npy", Q)
print("Saved policy_taxi.npy with shape", Q.shape)

## 8. Try this

1. **Fully random behaviour.** Use `eps_0 = eps_min = 1.0` (no decay). Does the greedy policy still deliver the passenger? Why does it take so much longer? (Hint: Q-learning is off-policy, but a random taxi rarely completes a delivery.)
2. **Fewer episodes.** How few episodes (`n_episodes`) are enough to get 100/100 successful deliveries?
3. **Discount factor.** Try `GAMMA = 0.99`. Does the policy change? Does the number of steps change?